# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

**Estrategia:** reutilizo las funciones de limpieza del lab anterior (en `data_cleaning.py`), limpio **cada fichero por separado** y después los concateno.

¿Por qué no concatenar primero y limpiar después? Porque los ficheros no tienen el mismo formato:
- `file3` llama a las columnas `State` y `Gender` (en vez de `ST` y `GENDER`) y las tiene en otro orden.
- En `file3` el CLV y las quejas ya son numéricos, mientras que en `file1`/`file2` son texto (`"697953.59%"`, `"1/0/00"`).

Limpiando cada uno antes, los tres acaban con las mismas columnas y los mismos tipos, y `pd.concat` los junta sin problemas.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import data_cleaning as dc

base_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
raw = {f: pd.read_csv(base_url + f) for f in ["file1.csv", "file2.csv", "file3.csv"]}

for name, d in raw.items():
    print(name, d.shape)
    print(list(d.columns), "\n")

file1.csv (4008, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount'] 

file2.csv (996, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class'] 

file3.csv (7070, 11)
['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class'] 



⚠️ **Hallazgo importante: el CLV no está en la misma escala.** En `file1` y `file2` el CLV viene con un `%` y sus valores son unas 100 veces más grandes que en `file3`:

In [2]:
clv = lambda s: pd.to_numeric(s.astype(str).str.replace("%", ""), errors="coerce")
pd.DataFrame({name: clv(d["Customer Lifetime Value"]).describe() for name, d in raw.items()}).round(2)

,file1.csv,file2.csv,file3.csv
count,1068.00,992.00,7070.00
mean,793690.26,765149.39,8028.80
std,643478.41,621465.29,6942.42
min,200435.07,200435.07,1898.01
25%,403408.00,392411.92,3981.07
50%,588174.24,562705.38,5782.10
75%,896287.20,869052.61,8990.00
max,5816655.35,4922143.07,83325.38


La mediana es ~588.000 en `file1` frente a ~5.800 en `file3`. El `%` indica que el valor está multiplicado por 100 (`697953.59%` = 6979.5359). Si solo quitáramos el `%`, al juntar los ficheros el CLV mezclaría dos escalas y cualquier media o percentil saldría mal. Por eso `format_data_types` **divide entre 100** el CLV que viene como texto con `%`.

In [3]:
clean_dfs = [dc.clean(d) for d in raw.values()]

# Comprobamos que los tres tienen exactamente las mismas columnas antes de concatenar
for d in clean_dfs:
    print(d.shape, sorted(d.columns) == sorted(clean_dfs[0].columns))

(1071, 11) True
(996, 11) True
(7070, 11) True


In [4]:
df = pd.concat(clean_dfs, ignore_index=True)
print(df.shape)
df.head()

(9137, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Unknown,Master,5882,0,1000,0,Personal Auto,Four-Door Car,3
1,QZ44356,Arizona,F,Bachelor,6980,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,12887,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,7646,0,106,0,Corporate Auto,SUV,530
4,GA49547,Washington,M,High School or Below,5363,36357,68,0,Personal Auto,Four-Door Car,17


Al juntar varias fuentes es habitual que haya **clientes repetidos** entre ficheros. Lo comprobamos:

In [5]:
print("Filas duplicadas exactas:", df.duplicated().sum())
print("IDs de cliente repetidos:", df["customer"].duplicated().sum())

Filas duplicadas exactas: 70
IDs de cliente repetidos: 81


In [6]:
# 1) Fuera duplicados exactos
df = df.drop_duplicates()

# 2) ¿Qué pasa con los IDs que siguen repetidos?
dup_ids = df[df["customer"].duplicated(keep=False)].sort_values("customer")
dup_ids

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
741,AC40767,Washington,M,Bachelor,2498,68041,6464,0,Personal Auto,Two-Door Car,166
2888,AC40767,Washington,M,Bachelor,2498,68041,64,0,Personal Auto,Two-Door Car,166
2675,BJ22117,Washington,M,Master,5133,53390,64,0,Personal Auto,Four-Door Car,279
1187,BJ22117,Washington,M,Master,5133,53390,6464,0,Personal Auto,Four-Door Car,279
3533,CI38330,Washington,M,Bachelor,2550,72217,64,0,Personal Auto,Four-Door Car,91
354,CI38330,Washington,M,Bachelor,2550,72217,6464,0,Personal Auto,Four-Door Car,91
1346,KN13494,Washington,F,Master,5182,97131,6464,0,Personal Auto,Four-Door Car,307
2278,KN13494,Washington,F,Master,5182,97131,64,0,Personal Auto,Four-Door Car,307
2407,MP96373,Washington,M,Master,5720,54585,71,1,Personal Auto,Two-Door Car,341
1679,MP96373,Washington,M,Master,5720,54585,7575,1,Personal Auto,Two-Door Car,341


Quedan unos pocos clientes que aparecen en dos ficheros con algún valor distinto. Por ejemplo, `AC40767` tiene una prima mensual de `6464` en un fichero y `64` en otro, lo que parece un error de tecleo. `file3` es el fichero más limpio y es el último que concatenamos, así que me quedo con la **última** aparición (`keep="last"`).

In [7]:
df = df.drop_duplicates(subset="customer", keep="last").reset_index(drop=True)

print(df.shape)
print("IDs repetidos:", df["customer"].duplicated().sum())
print("Nulos:", df.isnull().sum().sum())
df.dtypes

(9056, 11)
IDs repetidos: 0
Nulos: 0


customer                       str
state                          str
gender                         str
education                      str
customer_lifetime_value      int64
income                       int64
monthly_premium_auto         int64
number_of_open_complaints    int64
policy_type                    str
vehicle_class                  str
total_claim_amount           int64
dtype: object

In [8]:
df.to_csv("combined_insurance_data.csv", index=False)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [9]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
mkt = pd.read_csv(url)

# "unnamed:_0" es el índice antiguo guardado como columna: no aporta nada
mkt = mkt.drop(columns="unnamed:_0")
print(mkt.shape)
mkt.head()

(10910, 26)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,48029,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,0,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,22139,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,23675,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [10]:
print(mkt.isnull().sum().sum(), "nulos |", mkt.duplicated().sum(), "duplicados")
mkt["sales_channel"].value_counts()

0 nulos | 443 duplicados


sales_channel
Agent          4121
Branch         3022
Call Center    2141
Web            1626
Name: count, dtype: int64

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [11]:
# En este dataset no hay canal "mail": los canales son Agent, Branch, Call Center y Web.
# Para una aseguradora los ingresos son las PRIMAS que pagan los clientes (monthly_premium_auto).
# total_claim_amount es lo que la aseguradora PAGA en siniestros (un coste), lo incluyo para comparar.
revenue_by_channel = mkt.pivot_table(
    index="sales_channel",
    values=["monthly_premium_auto", "total_claim_amount"],
    aggfunc="sum",
).round(2).sort_values("monthly_premium_auto", ascending=False)

revenue_by_channel["pct_revenue"] = (
    revenue_by_channel["monthly_premium_auto"] / revenue_by_channel["monthly_premium_auto"].sum() * 100
).round(2)
revenue_by_channel

,monthly_premium_auto,total_claim_amount,pct_revenue
sales_channel,,,
Agent,386335,1810226.82,38.00
Branch,280953,1301204.00,27.63
Call Center,197970,926600.82,19.47
Web,151511,706600.04,14.90


In [12]:
# ¿Ganan más por cliente algunos canales, o solo traen más clientes?
mkt.pivot_table(index="sales_channel", values="monthly_premium_auto", aggfunc=["count", "mean"]).round(2)

,count,mean
,monthly_premium_auto,monthly_premium_auto
sales_channel,,
Agent,4121,93.75
Branch,3022,92.97
Call Center,2141,92.47
Web,1626,93.18


**Insights:**
- **Agent** es el canal que más ingresos genera (~38 % de las primas), seguido de Branch (~28 %), Call Center (~20 %) y Web (~15 %).
- La prima media por cliente es prácticamente igual en todos los canales (~93 $). La diferencia de ingresos viene del **volumen de clientes** que trae cada canal, no de que vendan pólizas más caras.
- Las reclamaciones (`total_claim_amount`) siguen la misma proporción, así que ningún canal atrae clientes especialmente "caros".
- Recomendación: el canal Agent es el motor de ventas. Web es el más pequeño; si su coste de adquisición es bajo, podría tener margen de crecimiento.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [13]:
clv_gender_edu = mkt.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean",
).round(2)
clv_gender_edu

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


**Insights:**
- Las diferencias son pequeñas: todas las medias están entre ~7.300 y ~8.700 $. Ni el género ni el nivel educativo son un factor determinante del CLV.
- El CLV medio más alto es el de las **mujeres con High School or Below** (~8.675 $) y el más bajo el de las personas con **Doctor** (~7.330 $ en mujeres y ~7.415 $ en hombres).
- Curiosamente, más estudios no significa más valor de cliente: "High School or Below" y "Master" están por encima de "Bachelor" y "Doctor".
- Ojo: la media es sensible a outliers (el CLV llega a 83.000 $). Antes de decidir nada habría que comparar también con la mediana (`aggfunc="median"`).

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [14]:
# 1) Tabla resumen en formato ANCHO (wide): una columna por mes
complaints_wide = mkt.pivot_table(
    index="policy_type",
    columns="month",
    values="number_of_open_complaints",
    aggfunc="sum",
).round(2)
complaints_wide

month,1,2
policy_type,,
Corporate Auto,443.43,385.21
Personal Auto,1727.61,1453.68
Special Auto,87.07,95.23


In [15]:
# 2) La pasamos a formato LARGO (long) con melt: una fila = (policy_type, month, complaints)
complaints_long = (
    complaints_wide.reset_index()
    .melt(id_vars="policy_type", var_name="month", value_name="number_of_complaints")
    .sort_values("number_of_complaints", ascending=False)
    .reset_index(drop=True)
)
complaints_long

,policy_type,month,number_of_complaints
0,Personal Auto,1,1727.61
1,Personal Auto,2,1453.68
2,Corporate Auto,1,443.43
3,Corporate Auto,2,385.21
4,Special Auto,2,95.23
5,Special Auto,1,87.07


In [16]:
# Forma alternativa de llegar directamente al formato largo: groupby + reset_index
mkt.groupby(["policy_type", "month"])["number_of_open_complaints"].sum().round(2).reset_index()

,policy_type,month,number_of_open_complaints
0,Corporate Auto,1,443.43
1,Corporate Auto,2,385.21
2,Personal Auto,1,1727.61
3,Personal Auto,2,1453.68
4,Special Auto,1,87.07
5,Special Auto,2,95.23


**Insights:**
- **Personal Auto** acumula la gran mayoría de las quejas en los dos meses, algo esperable porque es el tipo de póliza con más clientes.
- En Personal Auto y Corporate Auto hubo **más quejas en enero (mes 1)** que en febrero. Special Auto es la excepción, con un poco más en febrero.
- Para comparar de forma justa habría que normalizar por número de clientes (quejas por cliente), porque los totales dependen mucho del tamaño de cada grupo.